# 📡 Phase 2 — T7: Fading Uygulanmış Dataset Üretimi

Bu notebook, orijinal RML2016.10a (AWGN) dataseti üzerine **Rayleigh** ve **Rician** fading kanallarını uygulayarak yeni dataset'ler üretir ve Google Drive'a kaydeder.

**Üretilecek dosyalar:**

| Dosya | Kanal Modeli | Açıklama |
|-------|-------------|----------|
| `RML2016.10a_rayleigh.pkl` | Rayleigh | NLOS ortam, derin sönümleme |
| `RML2016.10a_rician_K3.pkl` | Rician (K=3) | LOS bileşeni orta düzey |
| `RML2016.10a_rician_K10.pkl` | Rician (K=10) | LOS bileşeni baskın |

**Adımlar:**
1. Ortam kurulumu ve orijinal dataset yükleme
2. Faded dataset üretimi (Rayleigh + Rician K=3, K=10)
3. Dataset'leri Google Drive'a kaydetme ve doğrulama

> **Not:** Görselleştirme ve analiz için → `05_visualize_fading_effects.ipynb`

## 1. Ortam Kurulumu

In [ ]:
import numpy as np
import pickle
import os
import time

print('✅ Kütüphaneler yüklendi.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/erigami-sl/AMR-UnderDifferentNoises-DL.git'
PROJECT_DIR = '/content/AMR-UnderDifferentNoises-DL'

if not os.path.exists(PROJECT_DIR):
    !git clone -b dev {REPO_URL}
else:
    !cd {PROJECT_DIR} && git pull origin dev
    print(f'Proje güncellendi: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
import sys
sys.path.insert(0, PROJECT_DIR)
print(f'✅ Çalışma dizini: {os.getcwd()}')

## 2. Orijinal Dataset Yükleme

In [ ]:
DRIVE_DATASET = '/content/drive/MyDrive/AMR-Project/RML2016.10a_dict.pkl'
LOCAL_DATASET = os.path.join(PROJECT_DIR, 'data', 'RML2016.10a_dict.pkl')

if os.path.exists(DRIVE_DATASET):
    DATASET_PATH = DRIVE_DATASET
elif os.path.exists(LOCAL_DATASET):
    DATASET_PATH = LOCAL_DATASET
else:
    raise FileNotFoundError(
        f'Dataset bulunamadı!\n'
        f'  Drive: {DRIVE_DATASET}\n'
        f'  Lokal: {LOCAL_DATASET}\n'
        f'Lütfen RML2016.10a_dict.pkl dosyasını yukarıdaki konumlardan birine yerleştirin.'
    )

Xd = pickle.load(open(DATASET_PATH, 'rb'), encoding='iso-8859-1')

print(f'📁 Dataset: {DATASET_PATH}')
print(f'📊 Anahtar sayısı: {len(Xd.keys())} (beklenen: 220)')
print(f'📐 Örnek shape: {list(Xd.values())[0].shape} (beklenen: (1000, 2, 128))')
print(f'📦 Toplam örnek: {sum(v.shape[0] for v in Xd.values()):,}')

## 3. Faded Dataset Üretimi

Her kanal modeli için `generate_faded_dataset()` çağrılır.  
Tekrarlanabilirlik için sabit `seed` kullanılır.

In [ ]:
from src.utils.channels import generate_faded_dataset

MASTER_SEED = 2016  # Projenin standart seed'i (config.py ile tutarlı)

CHANNEL_CONFIGS = {
    'rayleigh': {
        'channel_type': 'rayleigh',
        'K': None,
        'seed': MASTER_SEED,
        'filename': 'RML2016.10a_rayleigh.pkl',
        'label': 'Rayleigh Fading (NLOS)'
    },
    'rician_K3': {
        'channel_type': 'rician',
        'K': 3.0,
        'seed': MASTER_SEED + 1000,
        'filename': 'RML2016.10a_rician_K3.pkl',
        'label': 'Rician Fading (K=3, Moderate LOS)'
    },
    'rician_K10': {
        'channel_type': 'rician',
        'K': 10.0,
        'seed': MASTER_SEED + 2000,
        'filename': 'RML2016.10a_rician_K10.pkl',
        'label': 'Rician Fading (K=10, Strong LOS)'
    }
}

print('📋 Üretim planı:')
for name, cfg in CHANNEL_CONFIGS.items():
    K_str = f'K={cfg["K"]}' if cfg['K'] else 'N/A'
    print(f'  • {cfg["label"]:40s} → {cfg["filename"]}  (seed={cfg["seed"]}, {K_str})')

In [ ]:
faded_datasets = {}

for name, cfg in CHANNEL_CONFIGS.items():
    print(f'\n🔄 Üretiliyor: {cfg["label"]}...')
    
    kwargs = {'channel_type': cfg['channel_type'], 'seed': cfg['seed']}
    if cfg['K'] is not None:
        kwargs['K'] = cfg['K']
    
    t0 = time.time()
    faded_datasets[name] = generate_faded_dataset(Xd, **kwargs)
    elapsed = time.time() - t0
    
    n_keys = len(faded_datasets[name])
    has_nan = any(np.isnan(v).any() for v in faded_datasets[name].values())
    
    print(f'   ✅ Tamamlandı: {elapsed:.2f}s — {n_keys} anahtar')
    print(f'   🔍 NaN: {"❌ VAR!" if has_nan else "✅ Temiz"}')

print(f'\n🎉 Tüm dataset\'ler başarıyla üretildi!')

## 4. Kaydetme ve Doğrulama

In [ ]:
DRIVE_SAVE_DIR = '/content/drive/MyDrive/AMR-Project'
LOCAL_SAVE_DIR = os.path.join(PROJECT_DIR, 'data')
SAVE_DIR = DRIVE_SAVE_DIR if os.path.exists('/content/drive/MyDrive') else LOCAL_SAVE_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'💾 Kayıt dizini: {SAVE_DIR}\n')

for name, cfg in CHANNEL_CONFIGS.items():
    filepath = os.path.join(SAVE_DIR, cfg['filename'])
    with open(filepath, 'wb') as f:
        pickle.dump(faded_datasets[name], f, protocol=pickle.HIGHEST_PROTOCOL)
    size_mb = os.path.getsize(filepath) / 1024 / 1024
    print(f'  ✅ {cfg["filename"]:40s} — {size_mb:.1f} MB')

print(f'\n🎉 Tüm dosyalar kaydedildi!')

In [ ]:
# ── Kayıt doğrulaması ──
print('🔍 Doğrulama...\n')

for name, cfg in CHANNEL_CONFIGS.items():
    filepath = os.path.join(SAVE_DIR, cfg['filename'])
    Xd_loaded = pickle.load(open(filepath, 'rb'))
    
    assert len(Xd_loaded) == 220, f'{name}: Anahtar sayısı {len(Xd_loaded)} != 220'
    for key in Xd_loaded:
        assert Xd_loaded[key].shape == (1000, 2, 128), f'{name}/{key}: Shape hatası'
    assert not any(np.isnan(v).any() for v in Xd_loaded.values()), f'{name}: NaN!'
    assert not np.array_equal(Xd_loaded[('QPSK', 10)], Xd[('QPSK', 10)]), f'{name}: Fading uygulanmamış!'
    
    print(f'  ✅ {cfg["label"]:45s} — doğrulandı')

print('\n✅ Tüm dosyalar doğrulandı. Dataset\'ler eğitime hazır!')
print('📊 Görselleştirme için → notebooks/05_visualize_fading_effects.ipynb')

---
## ✅ T7 Tamamlandı

Fading uygulanmış dataset'ler üretildi ve Google Drive'a kaydedildi.

**Sonraki adımlar:**
- Görselleştirme → `05_visualize_fading_effects.ipynb`
- Model eğitimi → T9